# AIDev Project 2 — Exploration Notebook

### 1. Imports and config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')

# Hugging Face base path — pandas can read parquet directly from here
HF_BASE = 'hf://datasets/hao-li/AIDev/'

print('Setup complete.')

### 2. Load AIDev-pop tables
`pull_request`, `pr_reviews`, `pr_comments`.

In [ ]:
# Load AIDev-pop pull request table
pr_df = pd.read_parquet(HF_BASE + 'pull_request.parquet')
print(f'pull_request shape: {pr_df.shape}')
print(f'columns: {list(pr_df.columns)}')
pr_df.head(3)

In [ ]:
# Reviews table
reviews_df = pd.read_parquet(HF_BASE + 'pr_reviews.parquet')
print(f'pr_reviews shape: {reviews_df.shape}')
print(f'columns: {list(reviews_df.columns)}')
reviews_df.head(3)

In [ ]:
# Comments table (issue-style PR thread comments)
comments_df = pd.read_parquet(HF_BASE + 'pr_comments.parquet')
print(f'pr_comments shape: {comments_df.shape}')
print(f'columns: {list(comments_df.columns)}')
comments_df.head(3)

### 3. Schema summaries
Dtype, null %, unique count, and a sample value per column.

In [ ]:
def schema_summary(df, name):
    """Compact schema view: dtype, null count, unique count, sample value."""
    summary = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'nulls': df.isna().sum(),
        'null_pct': (df.isna().mean() * 100).round(1),
        'unique': df.nunique(dropna=True),
        'sample': [df[c].dropna().iloc[0] if df[c].dropna().size else None for c in df.columns]
    })
    print(f'\n=== {name} ({len(df):,} rows) ===')
    return summary

schema_summary(pr_df, 'pull_request')

In [ ]:
schema_summary(reviews_df, 'pr_reviews')

In [ ]:
schema_summary(comments_df, 'pr_comments')

### 4. Headline counts
Totals for the dataset slide: PRs, agents, repos, reviewers.

In [ ]:
# NOTE: column names below assume the standard AIDev schema. If a column isn't found,
# check the schema_summary output above and substitute the correct name.

headline = {
    'Total agent-authored PRs': len(pr_df),
    'Unique agents': pr_df['agent'].nunique() if 'agent' in pr_df.columns else 'check schema',
    'Unique repositories': pr_df['repo_id'].nunique() if 'repo_id' in pr_df.columns else pr_df.get('repository_id', pd.Series()).nunique(),
    'Total formal reviews': len(reviews_df),
    'Unique reviewers': reviews_df['user_id'].nunique() if 'user_id' in reviews_df.columns else 'check schema',
    'Total PR thread comments': len(comments_df),
}

for k, v in headline.items():
    print(f'{k:35s}: {v:,}' if isinstance(v, (int, np.integer)) else f'{k:35s}: {v}')

### 5. PRs per agent

In [ ]:
agent_col = 'agent' if 'agent' in pr_df.columns else 'agent_label'  # adjust if needed
agent_counts = pr_df[agent_col].value_counts()
print(agent_counts)

fig, ax = plt.subplots(figsize=(8, 4))
agent_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Number of PRs')
ax.set_title('AIDev-pop: PRs per AI coding agent')
plt.tight_layout()
plt.savefig('prs_per_agent.png', dpi=150, bbox_inches='tight')
plt.show()

### 6. PR outcomes
Merged / closed / open — overall and by agent.

In [ ]:
# GitHub PR state is usually 'open' or 'closed'; merged is a separate boolean.
# Combine into a clean outcome label.
def label_outcome(row):
    if row.get('merged') is True or row.get('merged_at') is not None and pd.notna(row.get('merged_at')):
        return 'merged'
    if row.get('state') == 'closed':
        return 'closed_unmerged'
    return 'open'

# Adjust based on actual columns from schema_summary above
if 'merged_at' in pr_df.columns:
    pr_df['outcome'] = np.where(
        pr_df['merged_at'].notna(), 'merged',
        np.where(pr_df['state'] == 'closed', 'closed_unmerged', 'open')
    )
elif 'merged' in pr_df.columns:
    pr_df['outcome'] = np.where(
        pr_df['merged'] == True, 'merged',
        np.where(pr_df['state'] == 'closed', 'closed_unmerged', 'open')
    )

outcome_counts = pr_df['outcome'].value_counts()
print(outcome_counts)
print(f'\nMerge rate: {outcome_counts.get("merged", 0) / len(pr_df) * 100:.1f}%')

In [ ]:
# Outcome by agent — interesting cross-tab
ct = pd.crosstab(pr_df[agent_col], pr_df['outcome'], normalize='index') * 100
print(ct.round(1))

fig, ax = plt.subplots(figsize=(9, 4))
ct.plot(kind='barh', stacked=True, ax=ax, colormap='RdYlGn_r')
ax.set_xlabel('% of PRs')
ax.set_title('PR outcomes by agent (AIDev-pop)')
ax.legend(title='Outcome', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('outcomes_by_agent.png', dpi=150, bbox_inches='tight')
plt.show()

### 7. Review intensity per PR
Review and comment counts joined onto each PR.

In [ ]:
# Reviews and comments use 'pr_id' to point to the PR;
# the PR table uses 'id' as its primary key. Two different keys.
reviews_per_pr = reviews_df.groupby('pr_id').size().rename('n_reviews')
comments_per_pr = comments_df.groupby('pr_id').size().rename('n_comments')

pr_enriched = (
    pr_df.set_index('id')
         .join(reviews_per_pr)
         .join(comments_per_pr)
         .fillna({'n_reviews': 0, 'n_comments': 0})
)
pr_enriched['n_reviews'] = pr_enriched['n_reviews'].astype(int)
pr_enriched['n_comments'] = pr_enriched['n_comments'].astype(int)

# Sanity check — these should both be well above zero
print(f"PRs with at least 1 review:  {(pr_enriched['n_reviews'] > 0).sum():,}")
print(f"PRs with at least 1 comment: {(pr_enriched['n_comments'] > 0).sum():,}")
print(f"\nReviews per PR — descriptive stats:")
print(pr_enriched['n_reviews'].describe().round(2))
print(f"\nComments per PR — descriptive stats:")
print(pr_enriched['n_comments'].describe().round(2))

# What % of PRs receive zero formal reviews? Zero comments?
print(f"\nPRs with 0 formal reviews:   {(pr_enriched['n_reviews'] == 0).mean() * 100:.1f}%")
print(f"PRs with 0 thread comments:  {(pr_enriched['n_comments'] == 0).mean() * 100:.1f}%")

In [ ]:
# Distribution plot — log scale because of long tails
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

pr_enriched['n_reviews'].clip(upper=20).hist(bins=21, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Reviews per PR (clipped at 20)')
axes[0].set_xlabel('# reviews'); axes[0].set_ylabel('# PRs')

pr_enriched['n_comments'].clip(upper=20).hist(bins=21, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Thread comments per PR (clipped at 20)')
axes[1].set_xlabel('# comments'); axes[1].set_ylabel('# PRs')

plt.tight_layout()
plt.savefig('review_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("No-review rate by agent")
print("=" * 60)

no_review_by_agent = (
    pr_enriched.assign(no_review=(pr_enriched['n_reviews'] == 0))
               .groupby('agent')
               .agg(
                   total_prs=('no_review', 'size'),
                   no_review_count=('no_review', 'sum'),
                   no_review_pct=('no_review', 'mean')
               )
)
no_review_by_agent['no_review_pct'] = (no_review_by_agent['no_review_pct'] * 100).round(1)
no_review_by_agent = no_review_by_agent.sort_values('no_review_pct', ascending=False)
print(no_review_by_agent)

# Visualize
fig, ax = plt.subplots(figsize=(8, 4))
no_review_by_agent['no_review_pct'].plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('% of PRs with zero formal reviews')
ax.set_title('No-review rate by agent (AIDev-pop)')
ax.axvline(76, color='gray', linestyle='--', alpha=0.6, label='Overall avg (~76%)')
ax.legend()
plt.tight_layout()
plt.savefig('no_review_by_agent.png', dpi=150, bbox_inches='tight')
plt.show()


print("What happens to PRs that receive zero reviews?")
print("=" * 60)

pr_enriched['no_review'] = pr_enriched['n_reviews'] == 0

# Outcome distribution: no-review vs. reviewed PRs
outcome_by_review = pd.crosstab(
    pr_enriched['no_review'].map({True: 'No reviews', False: 'Reviewed'}),
    pr_enriched['outcome'],
    normalize='index'
) * 100
print("\nOutcome % within each group:")
print(outcome_by_review.round(1))

# Visualize
fig, ax = plt.subplots(figsize=(9, 3.5))
outcome_by_review.plot(kind='barh', stacked=True, ax=ax, colormap='RdYlGn_r')
ax.set_xlabel('% of PRs')
ax.set_title('PR outcomes: no-review vs. reviewed PRs')
ax.legend(title='Outcome', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('no_review_vs_reviewed_outcomes.png', dpi=150, bbox_inches='tight')
plt.show()


print("The 'fast lane': PRs merged with no formal review")
print("=" * 60)

fast_lane = pr_enriched[(pr_enriched['n_reviews'] == 0) & (pr_enriched['outcome'] == 'merged')]
print(f"Fast-lane PRs: {len(fast_lane):,} ({len(fast_lane) / len(pr_enriched) * 100:.1f}% of all PRs)")
print(f"\nFast-lane PRs by agent:")
print(fast_lane['agent'].value_counts())

# Fast-lane share within each agent
fast_lane_pct = (
    pr_enriched.assign(fast=(pr_enriched['n_reviews'] == 0) & (pr_enriched['outcome'] == 'merged'))
               .groupby('agent')['fast'].mean() * 100
).round(1).sort_values(ascending=False)
print(f"\n% of each agent's PRs that hit the fast lane:")
print(fast_lane_pct)

In [ ]:
fast_lane_pct = (
    pr_enriched.assign(fast=(pr_enriched['n_reviews'] == 0) & (pr_enriched['outcome'] == 'merged'))
               .groupby('agent')['fast'].mean() * 100
).round(1).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(fast_lane_pct.index, fast_lane_pct.values, color='#2E7D32', edgecolor='white')

# Highlight Codex in a distinct color since it's the standout
for bar, agent in zip(bars, fast_lane_pct.index):
    if agent == 'OpenAI_Codex':
        bar.set_color('#C62828')

# Value labels at the end of each bar
for bar, val in zip(bars, fast_lane_pct.values):
    ax.text(val + 1, bar.get_y() + bar.get_height() / 2,
            f'{val}%', va='center', fontsize=11, fontweight='bold')

# Reference line at the dataset-wide average (53.8%)
ax.axvline(53.8, color='gray', linestyle='--', alpha=0.6, linewidth=1)
ax.text(53.8, 4.5, 'Overall: 53.8%', color='gray', fontsize=9, ha='center',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.8))

ax.set_xlabel('% of agent\'s PRs merged with zero formal reviews', fontsize=11)
ax.set_title('The "fast lane": agent PRs merged without review',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlim(0, max(fast_lane_pct.values) + 10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('fast_lane_by_agent.png', dpi=150, bbox_inches='tight')
plt.show()

print('Saved fast_lane_by_agent.png')

### 8. Time-to-decision
Hours from PR creation to merge or close, by agent and outcome.

In [ ]:
# Compute time-to-decision in hours.
# Adjust column names per the actual schema (created_at, closed_at, merged_at).
for col in ['created_at', 'closed_at', 'merged_at']:
    if col in pr_df.columns:
        pr_df[col] = pd.to_datetime(pr_df[col], errors='coerce', utc=True)

decision_time = pr_df['closed_at'].fillna(pr_df.get('merged_at')) - pr_df['created_at']
pr_df['decision_hours'] = decision_time.dt.total_seconds() / 3600

decided = pr_df[pr_df['outcome'] != 'open'].copy()
print('Time-to-decision (hours), by outcome:')
print(decided.groupby('outcome')['decision_hours'].describe(percentiles=[.25, .5, .75, .9]).round(1))

In [ ]:
# Visualize: log-scale boxplot of decision time by outcome and agent
plot_df = decided[decided['decision_hours'] > 0].copy()
plot_df['log_hours'] = np.log10(plot_df['decision_hours'])

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=plot_df, x=agent_col, y='log_hours', hue='outcome', ax=ax, showfliers=False)
ax.set_ylabel('log10(hours to decision)')
ax.set_title('Time to decision by agent and outcome')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('time_to_decision.png', dpi=150, bbox_inches='tight')
plt.show()

### 9. Review verdict mix
Distribution of `APPROVED` / `CHANGES_REQUESTED` / `COMMENTED` / `DISMISSED`.

In [ ]:
if 'state' in reviews_df.columns:
    verdict_col = 'state'
elif 'review_state' in reviews_df.columns:
    verdict_col = 'review_state'
else:
    verdict_col = None
    print('Check schema_summary for the verdict column name.')

if verdict_col:
    print(reviews_df[verdict_col].value_counts())
    fig, ax = plt.subplots(figsize=(7, 4))
    reviews_df[verdict_col].value_counts().plot(kind='barh', ax=ax, color='teal')
    ax.set_title('Distribution of review verdicts')
    ax.set_xlabel('# reviews')
    plt.tight_layout()
    plt.savefig('review_verdicts.png', dpi=150, bbox_inches='tight')
plt.show()

### 10. Save enriched PR table

In [ ]:
# Save the enriched PR table for downstream notebooks
pr_enriched.reset_index().to_parquet('pr_enriched.parquet', index=False)
print('Saved pr_enriched.parquet')